# BloodBridge AI — Data Preprocessing

In this notebook, we perform the baseline cleaning and preparation of our daily operational records. This includes:
- Ensuring data schema requirements are met.
- Dropping duplicate entries and handling missing values.
- Converting values to proper numeric types and boundary clipping (preventing negative operational volumes).
- Basic feature extraction (inventory ratios, day of the week, month indices).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

# Locate project root
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src").exists():
    ROOT = ROOT.parent
    
DATA_DIR = ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
print(f"Project root identified: {ROOT}")

Project root identified: C:\Users\dhair\OneDrive\Desktop\College\Project\project implementation part\BloodBridge_AI


### Load the Simulation Dataset

Let's load the generated raw synthetic transactional CSV file.

In [2]:
raw_path = INTERIM_DIR / "synthetic_daily_blood_inventory.csv"
raw_df = pd.read_csv(raw_path, parse_dates=["record_date"])
print(f"Loaded dataset with {len(raw_df):,} rows.")

Loaded dataset with 25,920 rows.


### Cleaning Schema Verification and Boundary Checks

In [3]:
required_columns = {
    "record_date", 
    "bank_id", 
    "blood_group", 
    "inventory_units", 
    "storage_capacity_units", 
    "donations_received", 
    "requests_received"
}

# Schema assertion
assert required_columns.issubset(raw_df.columns), "Schema is missing required columns"

# Drop duplicates and drop rows missing critical operational data
clean_df = raw_df.drop_duplicates().dropna(subset=list(required_columns)).copy()

# Ensure positive boundaries on numerical counts
numerical_cols = [
    "inventory_units", 
    "storage_capacity_units", 
    "donations_received", 
    "requests_received", 
    "units_expired"
]

for col in numerical_cols:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce").clip(lower=0)
    
clean_df = clean_df.dropna()
clean_df = clean_df.sort_values(["bank_id", "blood_group", "record_date"])
print(f"Rows remaining after validation/cleaning: {len(clean_df):,}")

Rows remaining after validation/cleaning: 25,920


### Feature Extraction: Ratios and Calendar Values

Now we calculate features like:
1. `inventory_ratio`: Current stock level relative to storage capacity.
2. `day_of_week`: Day index (Monday=0 to Sunday=6).
3. `month`: Month index (January=1 to December=12).

In [4]:
clean_df["inventory_ratio"] = clean_df["inventory_units"] / clean_df["storage_capacity_units"]
clean_df["day_of_week"] = clean_df["record_date"].dt.dayofweek
clean_df["month"] = clean_df["record_date"].dt.month

min_ratio = clean_df["inventory_ratio"].min()
max_ratio = clean_df["inventory_ratio"].max()
print(f"Processed dataset. Inventory ratio range: {min_ratio:.2f} - {max_ratio:.2f}")

Processed dataset. Inventory ratio range: 0.00 - 1.00


### Export Preprocessed Data

In [5]:
output_path = INTERIM_DIR / "preprocessed_synthetic_bloodbridge.csv"
clean_df.to_csv(output_path, index=False)
print(f"Saved clean dataset to {output_path.resolve()}")

Saved clean dataset to C:\Users\dhair\OneDrive\Desktop\College\Project\project implementation part\BloodBridge_AI\data\interim\preprocessed_synthetic_bloodbridge.csv
